# Урок 02 - Изучение Microsoft Agent Framework

**Microsoft Agent Framework (MAF)** — это единая платформа для создания ИИ-агентов. Она предоставляет чистую, композиционную архитектуру с четырьмя основными компонентами:

- **Client** – соединяется с конечной точкой модели ИИ и отвечает за коммуникацию
- **Agent** – оборачивает клиент с инструкциями и определениями инструментов
- **Tools** – расширяют возможности агента пользовательскими функциями, которые модель может вызывать
- **Session** – поддерживает историю диалога для многошаговых взаимодействий

В этом уроке мы создадим **агента для бронирования путешествий**, который проверяет доступность направления, используя эти концепции.


## Настройка


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q
! pip install python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

## Понимание архитектуры Agent Framework

Microsoft Agent Framework использует многослойную архитектуру:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Клиент** – `FoundryChatClient` подключается к развертыванию Azure OpenAI. Он отвечает за аутентификацию, форматирование запросов и разбор ответов.
2. **Агент** – Создается из клиента через `provider.create_agent()`, агент объединяет доступ к модели с инструкциями (системным приглашением) и инструментами.
3. **Инструменты** – Функции Python, украшенные `@tool`, которые агент может вызывать для выполнения действий или получения данных.
4. **Сессия** – Объект `AgentSession` (создаётся через `agent.create_session()`), который хранит историю разговора, обеспечивая многоходовый диалог, где агент помнит предыдущий контекст.

Давайте построим каждый слой шаг за шагом.


In [ ]:
# Create the client – this is the connection to the AI model
endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Добавление инструментов с помощью декоратора @tool

Инструменты позволяют агентам выполнять действия, выходящие за рамки генерации текста. Декоратор `@tool` преобразует обычную функцию Python в то, что агент может вызвать.

Основные моменты:
- Используйте `Annotated[type, "description"]`, чтобы модель понимала каждый параметр.
- Докстринг становится описанием инструмента, которое видит модель.
- `approval_mode="never_require"` означает, что инструмент запускается автоматически без подтверждения пользователя.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Создание агента с инструментами

Теперь мы объединяем клиент, инструкции и инструменты в агента. `instructions` выступают в роли системного подсказки — они определяют персонажа и поведение агента.


In [ ]:
agent = provider.as_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Многоходовые разговоры с сессиями

`AgentSession` (создаётся с помощью `agent.create_session()`) отслеживает все сообщения в разговоре. Передавая одну и ту же сессию в каждый вызов `agent.run()`, агент получает доступ ко всей истории переписки и может ссылаться на более ранние сообщения.

Мы передаём `tools=[check_destination_availability]`, чтобы агент мог вызывать наш проверщик доступности на каждом шаге.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Резюме

В этом уроке вы изучили четыре основных компонента Microsoft Agent Framework:

| Концепция | Чему вы научились |
|---------|------------------|
| **Клиент** | `FoundryChatClient` подключается к Azure OpenAI с аутентификацией по учетным данным |
| **Агент** | `provider.create_agent()` объединяет подключение к модели с инструкциями и именем |
| **Инструменты** | Декоратор `@tool` открывает функции Python для вызова агентом |
| **Сессия** | `agent.create_session()` поддерживает историю разговора в нескольких шагах |

Эти строительные блоки составляют основу для создания агентов, которые могут вести естественные беседы, вызывать внешние функции и поддерживать контекст — фундамент для более продвинутых паттернов агента в следующих уроках.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от ответственности**:
Этот документ был переведен с использованием сервиса машинного перевода [Co-op Translator](https://github.com/Azure/co-op-translator). Несмотря на наши усилия по обеспечению точности, имейте в виду, что автоматический перевод может содержать ошибки или неточности. Оригинальный документ на его исходном языке следует считать авторитетным источником. Для получения критически важной информации рекомендуется обратиться к профессиональному человеческому переводу. Мы не несем ответственности за любые недоразумения или неправильные толкования, возникшие в результате использования этого перевода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
